<a href="https://colab.research.google.com/github/surocham/tech_challenge_fase3_pnad_covid/blob/main/tech_challenge_fase3_pnad_covid_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tech Challenge Fase 3 — Analise PNAD-COVID 19

**Instituicao:** FIAP — Faculdade de Informatica e Administracao Paulista  
**Curso:** Pos-graduacao em Data Analytics  
**Autores:** Edgard Joseph Kiriyama e Matheus Pavani  
**Base de dados:** PNAD-COVID 19 — IBGE  
**Periodo analisado:** Setembro, Outubro e Novembro de 2020  

---

## Objetivo

Analisar o comportamento da populacao brasileira durante a pandemia de COVID-19, com base nos microdados da PNAD-COVID 19 do IBGE, identificando indicadores clinicos, comportamentais e economicos para subsidiar o planejamento hospitalar em caso de novo surto.

---

## Estrutura do Notebook

1. Instalacao e importacao de bibliotecas
2. Autenticacao e conexao com o Google BigQuery
3. Carregamento e tratamento dos dados
4. Upload para o banco de dados em nuvem
5. Analise exploratoria dos dados
6. Visualizacoes graficas
7. Conclusoes

---
## 1. Instalacao e Importacao de Bibliotecas

Instalacao das bibliotecas necessarias para o projeto:
- **pandas**: manipulacao e analise de dados
- **matplotlib / seaborn**: visualizacao grafica
- **plotly**: graficos interativos
- **google-cloud-bigquery / pandas-gbq**: conexao com o Google BigQuery
- **db-dtypes / openpyxl**: suporte a tipos de dados e arquivos Excel

In [ ]:
# Instalando as bibliotecas necessarias para o projeto
!pip install pandas matplotlib seaborn plotly google-cloud-bigquery db-dtypes openpyxl pandas-gbq

In [ ]:
# Importando as bibliotecas
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import pandas_gbq
from google.cloud import bigquery
from google.colab import auth

print("Bibliotecas importadas com sucesso!")

---
## 2. Autenticacao e Conexao com o Google BigQuery

O Google BigQuery e o banco de dados em nuvem utilizado para armazenar e consultar os microdados da PNAD-COVID 19. A autenticacao e realizada com a conta Google associada ao projeto na Google Cloud Platform.

**Projeto GCP:** pnad-covid-19-494513  
**Dataset:** pnad_covid  
**Tabela:** microdados

In [ ]:
# Autenticando com o Google Cloud
auth.authenticate_user()
print("Autenticacao concluida!")

In [ ]:
# Conectando ao projeto do BigQuery
PROJECT_ID = "pnad-covid-19-494513"

client = bigquery.Client(project=PROJECT_ID)
print(f"Conectado ao projeto: {PROJECT_ID}")

---
## 3. Carregamento e Tratamento dos Dados

### 3.1 Variaveis Selecionadas

Foram selecionadas 20 variaveis organizadas em tres eixos tematicos:

**Sintomas Clinicos:** UF, B0011 (febre), B0012 (tosse), B0013 (dif. respirar), B0014 (dor cabeca), B0015 (dor peito), B002 (algum sintoma), B0103 (foi ao hospital)  
**Comportamento:** A002 (idade), A003 (sexo), A004 (cor/raca), B006 (ficou em casa), B009B (tipo estabelecimento), B0101 (posto saude), B011 (plano saude)  
**Economia:** C001 (trabalhou), C002 (afastado), C007 (setor), C01012 (renda), D0011 (moradores)

In [ ]:
# Verificando se os arquivos foram carregados corretamente no Colab
arquivos = [
    'PNAD_COVID_092020.csv',
    'PNAD_COVID_102020.csv',
    'PNAD_COVID_112020.csv'
]

for arquivo in arquivos:
    if os.path.exists(arquivo):
        tamanho = os.path.getsize(arquivo) / (1024 * 1024)
        print(f"{arquivo} encontrado! Tamanho: {tamanho:.1f} MB")
    else:
        print(f"{arquivo} NAO encontrado. Faca o upload do arquivo.")

In [ ]:
# Definindo as 20 variaveis selecionadas para o projeto
colunas = [
    'UF', 'A002', 'A003', 'A004',
    'B0011', 'B0012', 'B0013', 'B0014', 'B0015',
    'B002', 'B0103', 'B006', 'B009B', 'B0101', 'B011',
    'C001', 'C002', 'C007', 'C01012', 'D0011'
]

print(f"Total de variaveis selecionadas: {len(colunas)}")
print("Variaveis:", colunas)

In [ ]:
# Carregando os tres meses de dados
print("Carregando setembro...")
df_set = pd.read_csv('PNAD_COVID_092020.csv', sep=',', usecols=colunas)
df_set['mes'] = 'Setembro'
df_set['mes_num'] = 9
print(f"Setembro: {len(df_set):,} entrevistados")

print("Carregando outubro...")
df_out = pd.read_csv('PNAD_COVID_102020.csv', sep=',', usecols=colunas)
df_out['mes'] = 'Outubro'
df_out['mes_num'] = 10
print(f"Outubro: {len(df_out):,} entrevistados")

print("Carregando novembro...")
df_nov = pd.read_csv('PNAD_COVID_112020.csv', sep=',', usecols=colunas)
df_nov['mes'] = 'Novembro'
df_nov['mes_num'] = 11
print(f"Novembro: {len(df_nov):,} entrevistados")

# Unindo os tres meses em um unico DataFrame
df = pd.concat([df_set, df_out, df_nov], ignore_index=True)
print(f"\nTotal geral: {len(df):,} entrevistados nos tres meses")
print(f"Formato da tabela: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

### 3.2 Analise Exploratoria Inicial

Antes de qualquer analise, e necessario entender a estrutura dos dados, identificar valores ausentes e compreender os tipos de cada variavel.

In [ ]:
# Visualizando as primeiras linhas da tabela
print("Primeiras 5 linhas da tabela:")
df.head()

In [ ]:
# Verificando tipos de dados e valores ausentes
print("=== TIPOS DE DADOS ===")
print(df.dtypes)

print("\n=== VALORES AUSENTES POR COLUNA ===")
ausentes = df.isnull().sum()
percentual = (ausentes / len(df) * 100).round(2)
resumo = pd.DataFrame({
    'Valores ausentes': ausentes,
    'Percentual (%)': percentual
})
print(resumo[resumo['Valores ausentes'] > 0].sort_values('Percentual (%)', ascending=False))
print("\nNota: valores ausentes sao esperados pois o questionario usa perguntas condicionais.")
print("Ex: perguntas sobre sintomas so sao feitas para quem teve algum sintoma.")

### 3.3 Tratamento dos Dados

Etapas de tratamento realizadas:
- Preenchimento de valores ausentes nas colunas de sintomas com 0 (nao teve o sintoma)
- Criacao de variaveis derivadas: faixa etaria, sexo legivel, cor/raca legivel e regiao geografica

In [ ]:
# Tratamento dos dados

# 1. Preenchendo ausentes nas colunas de sintomas com 0
colunas_sintomas = ['B0011','B0012','B0013','B0014','B0015']
df[colunas_sintomas] = df[colunas_sintomas].fillna(0)

# 2. Criando coluna de faixa etaria
def faixa_etaria(idade):
    if idade <= 9:
        return '0 a 9 anos'
    elif idade <= 19:
        return '10 a 19 anos'
    elif idade <= 29:
        return '20 a 29 anos'
    elif idade <= 59:
        return '30 a 59 anos'
    else:
        return '60 anos ou mais'

df['faixa_etaria'] = df['A002'].apply(faixa_etaria)

# 3. Criando coluna de sexo legivel
df['sexo'] = df['A003'].map({1: 'Homem', 2: 'Mulher'})

# 4. Criando coluna de cor/raca legivel
df['cor_raca'] = df['A004'].map({
    1: 'Branca',
    2: 'Preta',
    3: 'Amarela',
    4: 'Parda',
    5: 'Indigena'
})

# 5. Criando coluna de regiao geografica
df['regiao'] = df['UF'].apply(lambda x:
    'Norte' if x in [11,12,13,14,15,16,17] else
    'Nordeste' if x in [21,22,23,24,25,26,27,28,29] else
    'Sudeste' if x in [31,32,33,35] else
    'Sul' if x in [41,42,43] else
    'Centro-Oeste'
)

print("Tratamento concluido!")
print(f"\nDistribuicao por regiao:")
print(df['regiao'].value_counts())
print(f"\nDistribuicao por sexo:")
print(df['sexo'].value_counts())

---
## 4. Upload para o Banco de Dados em Nuvem (Google BigQuery)

Os dados tratados sao enviados para o Google BigQuery, onde ficam armazenados permanentemente e disponiveis para consultas SQL. Isso constitui o banco de dados em nuvem exigido pelo projeto.

**Estrutura no BigQuery:**
```
Projeto: pnad-covid-19-494513
  Dataset: pnad_covid
    Tabela: microdados
```

In [ ]:
# Enviando os dados para o BigQuery
print("Iniciando upload para o BigQuery...")
print("Isso pode demorar alguns minutos...")

pandas_gbq.to_gbq(
    df,
    destination_table='pnad_covid.microdados',
    project_id=PROJECT_ID,
    if_exists='replace',
    progress_bar=True
)

print("\nUpload concluido com sucesso!")
print(f"Tabela disponivel em: {PROJECT_ID}.pnad_covid.microdados")

### 4.1 Consultas SQL no BigQuery

Com os dados no BigQuery, e possivel executar consultas SQL diretamente do Colab para análise dos dados.

In [ ]:
# Consulta SQL 1 - Sintomas por mes
query_sintomas = """
SELECT
  mes,
  mes_num,
  COUNT(*) AS total_entrevistados,
  ROUND(AVG(A002), 1) AS idade_media,
  COUNTIF(B0011 = 1) AS total_com_febre,
  COUNTIF(B0012 = 1) AS total_com_tosse,
  COUNTIF(B0013 = 1) AS total_com_dif_respirar
FROM `pnad-covid-19-494513.pnad_covid.microdados`
GROUP BY mes, mes_num
ORDER BY mes_num
"""

resultado_sintomas = client.query(query_sintomas).to_dataframe()
print("Consulta 1 - Sintomas por mes:")
print(resultado_sintomas.to_string(index=False))

In [ ]:
# Consulta SQL 2 - Perfil economico por mes
query_economia = """
SELECT
  mes,
  COUNTIF(C001 = 1) AS trabalharam,
  COUNTIF(C001 = 2) AS nao_trabalharam,
  COUNTIF(C002 = 1) AS afastados,
  ROUND(AVG(CASE WHEN C01012 > 0 THEN C01012 END), 2) AS renda_media,
  COUNTIF(B011 = 1) AS tem_plano_saude,
  COUNTIF(B011 = 2) AS nao_tem_plano_saude
FROM `pnad-covid-19-494513.pnad_covid.microdados`
GROUP BY mes, mes_num
ORDER BY mes_num
"""

resultado_economia = client.query(query_economia).to_dataframe()
print("Consulta 2 - Perfil economico por mes:")
print(resultado_economia.to_string(index=False))

In [ ]:
# Consulta SQL 3 - Sintomas por regiao
query_regiao = """
SELECT
  regiao,
  mes,
  COUNT(*) AS total,
  COUNTIF(B0011 = 1) AS febre,
  COUNTIF(B0012 = 1) AS tosse,
  COUNTIF(B0013 = 1) AS dif_respirar,
  COUNTIF(B0103 = 1) AS foram_ao_hospital
FROM `pnad-covid-19-494513.pnad_covid.microdados`
GROUP BY regiao, mes, mes_num
ORDER BY mes_num, regiao
"""

resultado_regiao = client.query(query_regiao).to_dataframe()
print("Consulta 3 - Sintomas por regiao:")
print(resultado_regiao.to_string(index=False))

---
## 5. Visualizacoes Graficas

Esta secao apresenta os graficos gerados para a analise dos tres eixos tematicos do projeto: sintomas clinicos, comportamento da populacao e caracteristicas economicas.

In [ ]:
# Configuracao visual dos graficos
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Paleta de cores do projeto
AZUL    = '#2980b9'
VERMELHO = '#e74c3c'
VERDE   = '#27ae60'
LARANJA = '#e67e22'
ROXO    = '#8e44ad'
CINZA   = '#95a5a6'

print("Configuracao visual pronta!")

### Grafico 1 — Prevalencia dos Sintomas por Mes

Analisa a proporcao da populacao que apresentou cada sintoma nos tres meses.

In [ ]:
# GRAFICO 1 - Prevalencia dos sintomas por mes
fig, ax = plt.subplots(figsize=(12, 6))

meses_ordem = ['Setembro', 'Outubro', 'Novembro']
sintomas = {
    'Febre': 'B0011',
    'Tosse': 'B0012',
    'Dif. Respirar': 'B0013',
    'Dor de Cabeca': 'B0014',
    'Dor no Peito': 'B0015'
}
cores = [AZUL, VERMELHO, VERDE, LARANJA, ROXO]
posicoes = [-2, -1, 0, 1, 2]
largura = 0.15
x = range(len(meses_ordem))

for i, (nome, col) in enumerate(sintomas.items()):
    valores = []
    for mes in meses_ordem:
        df_mes = df[df['mes'] == mes]
        percentual = (df_mes[col] == 1).mean() * 100
        valores.append(round(percentual, 2))
    ax.bar(
        [p + posicoes[i] * largura for p in x],
        valores, largura,
        label=nome, color=cores[i], alpha=0.85
    )

ax.set_xticks(x)
ax.set_xticklabels(meses_ordem)
ax.set_ylabel('Percentual da populacao (%)')
ax.set_title('Prevalencia dos Sintomas de COVID-19 por Mes\nBrasil, Setembro a Novembro de 2020')
ax.legend(loc='upper right')
ax.set_ylim(0, 3)

plt.tight_layout()
plt.savefig('grafico_01_sintomas_por_mes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico 1 salvo!")

### Grafico 2 — Distribuicao de Casos por Sexo

Analisa a proporcao de casos de febre entre homens e mulheres nos tres meses.

In [ ]:
# GRAFICO 2 - Distribuicao de sintomas por sexo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, mes in enumerate(['Setembro', 'Outubro', 'Novembro']):
    df_mes = df[df['mes'] == mes]
    contagem = df_mes[df_mes['B0011'] == 1]['sexo'].value_counts()
    axes[i].pie(
        contagem.values,
        labels=contagem.index,
        autopct='%1.1f%%',
        colors=[AZUL, VERMELHO],
        startangle=90
    )
    axes[i].set_title(f'{mes}')

fig.suptitle(
    'Distribuicao de Casos de Febre por Sexo\nBrasil, Setembro a Novembro de 2020',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('grafico_02_sintomas_por_sexo.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico 2 salvo!")

### Grafico 3 — Casos por Faixa Etaria

Analisa a distribuicao de casos de febre por grupo de idade.

In [ ]:
# GRAFICO 3 - Sintomas por faixa etaria
ordem_faixas = ['0 a 9 anos', '10 a 19 anos', '20 a 29 anos', '30 a 59 anos', '60 anos ou mais']

fig, ax = plt.subplots(figsize=(12, 6))

for i, mes in enumerate(['Setembro', 'Outubro', 'Novembro']):
    df_mes = df[(df['mes'] == mes) & (df['B0011'] == 1)]
    contagem = df_mes['faixa_etaria'].value_counts()
    contagem = contagem.reindex(ordem_faixas, fill_value=0)
    x = range(len(ordem_faixas))
    offset = (i - 1) * 0.25
    ax.bar([p + offset for p in x], contagem.values, 0.25,
           label=mes, color=[AZUL, VERDE, LARANJA][i], alpha=0.85)

ax.set_xticks(range(len(ordem_faixas)))
ax.set_xticklabels(ordem_faixas, rotation=15)
ax.set_ylabel('Numero de casos')
ax.set_title('Casos de Febre por Faixa Etaria e Mes\nBrasil, Setembro a Novembro de 2020')
ax.legend()
plt.tight_layout()
plt.savefig('grafico_03_faixa_etaria.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico 3 salvo!")

### Grafico 4 — Casos por Grande Regiao

Analisa a distribuicao geografica dos casos de febre por grande regiao do Brasil.

In [ ]:
# GRAFICO 4 - Sintomas por regiao
regioes = ['Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste']

fig, ax = plt.subplots(figsize=(12, 6))

for i, mes in enumerate(['Setembro', 'Outubro', 'Novembro']):
    df_mes = df[(df['mes'] == mes) & (df['B0011'] == 1)]
    contagem = df_mes['regiao'].value_counts()
    contagem = contagem.reindex(regioes, fill_value=0)
    x = range(len(regioes))
    offset = (i - 1) * 0.25
    ax.bar([p + offset for p in x], contagem.values, 0.25,
           label=mes, color=[AZUL, VERDE, LARANJA][i], alpha=0.85)

ax.set_xticks(range(len(regioes)))
ax.set_xticklabels(regioes)
ax.set_ylabel('Numero de casos')
ax.set_title('Casos de Febre por Regiao e Mes\nBrasil, Setembro a Novembro de 2020')
ax.legend()
plt.tight_layout()
plt.savefig('grafico_04_regioes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico 4 salvo!")

### Grafico 5 — Renda Media e Situacao de Trabalho

Analisa as caracteristicas economicas da populacao nos tres meses.

In [ ]:
# GRAFICO 5 - Renda media por mes
fig, ax = plt.subplots(figsize=(8, 6))

meses_ordem = ['Setembro', 'Outubro', 'Novembro']
renda_media = []

for mes in meses_ordem:
    df_mes = df[(df['mes'] == mes) & (df['C01012'] > 0)]
    renda_media.append(round(df_mes['C01012'].mean(), 2))

bars = ax.bar(meses_ordem, renda_media, color=[AZUL, VERDE, LARANJA], alpha=0.85)
ax.set_title('Renda Media dos Trabalhadores\nBrasil, Setembro a Novembro de 2020')
ax.set_ylabel('Renda media (R$)')
ax.set_ylim(0, max(renda_media) * 1.2)
for i, v in enumerate(renda_media):
    ax.text(i, v + 30, f'R$ {v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_05_renda_media.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico 5 salvo!")

In [ ]:
# GRAFICO 5b - Situacao de trabalho por mes
fig, ax = plt.subplots(figsize=(8, 6))

meses_ordem = ['Setembro', 'Outubro', 'Novembro']
trabalharam = []
nao_trabalharam = []

for mes in meses_ordem:
    df_mes = df[df['mes'] == mes]
    total = len(df_mes)
    trabalharam.append(round((df_mes['C001'] == 1.0).sum() / total * 100, 1))
    nao_trabalharam.append(round((df_mes['C001'] == 2.0).sum() / total * 100, 1))

x = range(len(meses_ordem))
ax.bar(x, trabalharam, label='Trabalharam', color=AZUL, alpha=0.85, width=0.4)
ax.bar(x, nao_trabalharam, bottom=trabalharam, label='Nao trabalharam', color=VERMELHO, alpha=0.85, width=0.4)
ax.set_xticks(x)
ax.set_xticklabels(meses_ordem)
ax.set_title('Situacao de Trabalho por Mes\nBrasil, Setembro a Novembro de 2020')
ax.set_ylabel('Percentual (%)')
ax.set_ylim(0, 100)
ax.legend()

for i, (t, nt) in enumerate(zip(trabalharam, nao_trabalharam)):
    ax.text(i, t / 2, f'{t}%', ha='center', va='center', color='white', fontweight='bold')
    ax.text(i, t + nt / 2, f'{nt}%', ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_05b_trabalho.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico 5b salvo!")

### Grafico 6 — Plano de Saude e Busca por Hospital

Analisa a cobertura de plano de saude e o percentual de pessoas que buscou atendimento hospitalar.

In [ ]:
# GRAFICO 6 - Plano de saude vs busca por hospital
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

meses_ordem = ['Setembro', 'Outubro', 'Novembro']
com_plano = []
sem_plano = []

for mes in meses_ordem:
    df_mes = df[(df['mes'] == mes) & (df['B011'].isin([1, 2]))]
    total = len(df_mes)
    com_plano.append(round((df_mes['B011'] == 1).sum() / total * 100, 1))
    sem_plano.append(round((df_mes['B011'] == 2).sum() / total * 100, 1))

x = range(len(meses_ordem))
axes[0].bar(x, com_plano, label='Com plano', color=AZUL, alpha=0.85, width=0.4)
axes[0].bar(x, sem_plano, bottom=com_plano, label='Sem plano', color=VERMELHO, alpha=0.85, width=0.4)
axes[0].set_xticks(x)
axes[0].set_xticklabels(meses_ordem)
axes[0].set_title('Cobertura de Plano de Saude\nBrasil, Setembro a Novembro de 2020')
axes[0].set_ylabel('Percentual (%)')
axes[0].set_ylim(0, 110)
axes[0].legend()
for i, (c, s) in enumerate(zip(com_plano, sem_plano)):
    axes[0].text(i, c / 2, f'{c}%', ha='center', va='center', color='white', fontweight='bold')
    axes[0].text(i, c + s / 2, f'{s}%', ha='center', va='center', color='white', fontweight='bold')

foi_hospital = []
for mes in meses_ordem:
    df_mes = df[df['mes'] == mes]
    total = len(df_mes)
    foi_hospital.append(round((df_mes['B0103'] == 1).sum() / total * 100, 2))

axes[1].bar(meses_ordem, foi_hospital, color=[AZUL, VERDE, LARANJA], alpha=0.85)
axes[1].set_title('Percentual que Foi ao Hospital\nBrasil, Setembro a Novembro de 2020')
axes[1].set_ylabel('Percentual (%)')
axes[1].set_ylim(0, max(foi_hospital) * 1.3)
for i, v in enumerate(foi_hospital):
    axes[1].text(i, v + 0.05, f'{v}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_06_plano_hospital.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico 6 salvo!")

### Grafico 7 — Casos por Cor e Raca

Analisa a distribuicao de casos de febre por cor ou raca da populacao.

In [ ]:
# GRAFICO 7 - Sintomas por cor e raca
racas = ['Branca', 'Preta', 'Parda', 'Amarela', 'Indigena']

fig, ax = plt.subplots(figsize=(12, 6))

for i, mes in enumerate(['Setembro', 'Outubro', 'Novembro']):
    df_mes = df[(df['mes'] == mes) & (df['B0011'] == 1)]
    contagem = df_mes['cor_raca'].value_counts()
    contagem = contagem.reindex(racas, fill_value=0)
    x = range(len(racas))
    offset = (i - 1) * 0.25
    ax.bar([p + offset for p in x], contagem.values, 0.25,
           label=mes, color=[AZUL, VERDE, LARANJA][i], alpha=0.85)

ax.set_xticks(range(len(racas)))
ax.set_xticklabels(racas)
ax.set_ylabel('Numero de casos')
ax.set_title('Casos de Febre por Cor e Raca\nBrasil, Setembro a Novembro de 2020')
ax.legend()
plt.tight_layout()
plt.savefig('grafico_07_cor_raca.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico 7 salvo!")

---
## 6. Conclusões

Com base na analise dos microdados da PNAD-COVID 19 referentes aos meses de setembro, outubro e novembro de 2020, os principais achados foram:

**Sintomas clinicos:**
- A dor no peito foi o sintoma com maior prevalencia nos tres meses
- A tosse e a dificuldade para respirar indicam forte demanda por suporte respiratorio
- Novembro mostrou leve crescimento em varios sintomas, sinalizando inicio da segunda onda

**Comportamento da populacao:**
- Mulheres representaram mais de 53% dos casos de febre
- A faixa de 30 a 59 anos concentrou o maior numero absoluto de casos
- O Nordeste liderou em numero de casos, seguido do Sudeste
- A populacao parda foi a mais afetada em numeros absolutos

**Caracteristicas economicas:**
- Apenas 37% da populacao trabalhou formalmente no periodo
- Renda media dos trabalhadores foi de aproximadamente R$ 2.224
- Mais de 90% da populacao nao possuia plano de saude privado
- Cerca de 5% buscou atendimento hospitalar em cada mes

**Recomendacoes ao hospital:**
1. Ampliar capacidade de atendimento respiratorio
2. Preparar para alta demanda do SUS (90% sem plano privado)
3. Protocolos especificos para idosos e populacao vulneravel
4. Implementar sistema de monitoramento continuo de indicadores
5. Atencao especial para regioes Nordeste e Sudeste

In [ ]:
# Listando todos os graficos salvos
graficos = [f for f in os.listdir() if f.startswith('grafico')]
print("Graficos disponiveis para download:")
for g in sorted(graficos):
    print(f"  {g}")
print(f"\nTotal de graficos gerados: {len(graficos)}")